In [1]:
from kloppy import sportec, statsbomb
from kloppy.domain.models import PitchDimensions, Dimension
from databallpy import get_game_from_kloppy

print("Parsing local XML tracking and JSON event files via Kloppy...")

# 1. Load Local Sportec Tracking Data
tracking_dataset = sportec.load_tracking(
    raw_data="hdfc_tracking_event_matches/tracking/positional_data_raw/MLS-COM-000001_MLS-SEA-0001KA_MLS-MAT-0009B7.xml",
    meta_data="hdfc_tracking_event_matches/tracking/match_information/MLS-COM-000001_MLS-SEA-0001KA_MLS-MAT-0009B7.xml",
    coordinates="sportec" 
)

# 3. Load Local StatsBomb Event Data 
event_dataset = statsbomb.load(
    event_data="hdfc_tracking_event_matches/events/4037572.json",
    lineup_data="hdfc_tracking_event_matches/lineups/4037572.json"
)

# --- FIX: Define an identical custom pitch dimensions structure for both ---
# Rather than manually creating PitchDimensions, we can copy the dimensions 
# from the StatsBomb dataset OR force both to use StatsBomb's exact coordinate system.
# The safest approach for databallpy is transforming BOTH to ensure metadata parity.

tracking_dataset = tracking_dataset.transform(to_coordinate_system="statsbomb")
event_dataset = event_dataset.transform(to_coordinate_system="statsbomb")

# Explicitly align the pitch metadata attributes so databallpy's strict check passes
tracking_dataset.metadata.pitch_dimensions = event_dataset.metadata.pitch_dimensions

print("Rescaled both datasets to identical coordinate systems and metadata.")

# 4. Transform and combine into a DataBallPy Game object
game = get_game_from_kloppy(
    tracking_dataset=tracking_dataset, 
    event_dataset=event_dataset
)

print("\n--- MATCH COMBINATION COMPLETED ---")
print(f"Tracking Dataset Frames: {len(game.tracking_data)}")
print(f"Event Dataset Logs: {len(game.event_data)}")

Parsing local XML tracking and JSON event files via Kloppy...
Rescaled both datasets to identical coordinate systems and metadata.


/Users/mbasurto/Documents/Projects/databallpy/.venv-313/lib/python3.13/site-packages/databallpy/utils/get_game.py:899: UserWarning: Game dates in kloppy TrackingDataset and EventDataset are not equal. Setting both to pd.Timestamp('1975-01-01').
  warnings.warn(



--- MATCH COMBINATION COMPLETED ---
Tracking Dataset Frames: 166538
Event Dataset Logs: 3455


In [2]:
# Sync the event and tracking data
game.synchronise_tracking_and_event_data()
# Combined data can be accessed via game.tracking_data and game.event_data DataFrames
print("\n--- SYNCHRONIZATION COMPLETED ---")
# Overall sync certainty
print(f"Sync Certainty: {game.tracking_data['sync_certainty'].mean():.4f}")
# Print certainty variance to check for consistency
print(f"Sync Certainty Variance: {game.tracking_data['sync_certainty'].var():.6f}")


--- SYNCHRONIZATION COMPLETED ---
Sync Certainty: 0.9454
Sync Certainty Variance: 0.009199


In [7]:
import copy
import itertools
import numpy as np
import pandas as pd

# Import the module where the function lives
import databallpy.utils.synchronise_tracking_and_event_data as sync_module

# =====================================================================
# PASS 1: Identify Bad Syncs from First Pass
# =====================================================================

events = ["shot", "dribble", "pass"]
key_events_df = game.event_data[
    game.event_data["databallpy_event"].isin(events)
].copy()


# Extract tracking frames from the first pass sync results
def extract_frame(event_id):
    try:
        return game.get_event_frame(event_id)["frame"].iloc[0]
    except (IndexError, ValueError):
        return np.nan


key_events_df["tracking_frame"] = (
    key_events_df["event_id"].apply(extract_frame).astype("Int64")
)

# Filter out unmatched events
events_df = key_events_df.dropna(subset=["tracking_frame"]).copy()
events_df["tracking_frame"] = events_df["tracking_frame"].astype(int)

# Extract ball tracking coordinates
ball_tracking = game.tracking_data[["frame", "ball_x", "ball_y"]].rename(
    columns={"frame": "tracking_frame"}
)

# Merge and calculate Euclidean distances
merged_df = pd.merge(events_df, ball_tracking, on="tracking_frame", how="left")
merged_df = merged_df.loc[:, ~merged_df.columns.duplicated(keep="last")]

merged_df["ball_to_event_distance"] = np.sqrt(
    (merged_df["ball_x"] - merged_df["start_x"]) ** 2
    + (merged_df["ball_y"] - merged_df["start_y"]) ** 2
)

# Flag distance discrepancies > 20 meters
merged_df["distance_flag"] = merged_df["ball_to_event_distance"] > 20
flagged_events = merged_df[merged_df["distance_flag"]]

# Get the list of unique IDs for the bad syncs
bad_event_ids = flagged_events["event_id"].tolist()
print(f"Number of poorly synchronized events identified: {len(bad_event_ids)}")
print(f"Average Sync Certainty of poor events: {flagged_events['sync_certainty'].mean()}")

Number of poorly synchronized events identified: 13
Average Sync Certainty of poor events: 0.5306758325953296


In [8]:
# =====================================================================
# PASS 2: Isolate Bad Syncs & Run Monkey-Patched Grid Search
# =====================================================================

# 1. Create an isolated deep copy of the game object
bad_game = copy.deepcopy(game)

# 2. Overwrite event data so it ONLY contains the poorly synced events
bad_game.event_data = game.event_data[
    game.event_data["event_id"].isin(bad_event_ids)
].reset_index(drop=True)

# Save a reference to the original function so we don't break anything permanently
original_combine_cost_functions = sync_module.combine_cost_functions

# --- STEP 2A: Define your Grid Search Space ---
grid_space = {
    "time_cost": [0.25, 0.5, 1.0, 1.5, 2.0],
    "ball_event_dist_cost": [1.0, 1.5, 2.0],
    "ball_player_dist_cost": [0.5, 1.0, 1.5, 2.0],
    "ball_acc_cost": [1.0],
    "player_ball_dist_inc_cost": [1.0, 2.0],
    "goal_angle_cost": [1.0],
}

# Generate all combinations
keys, values = zip(*grid_space.items())
experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"Total grid search iterations to run on bad syncs: {len(experiments)}")

# --- STEP 2B: Run the Grid Search on bad_game ---
results = []

for i, custom_weights in enumerate(experiments):
    print(
        f"Running iteration {i+1}/{len(experiments)} with weights: {custom_weights}"
    )

    # Define a wrapper function that forces your custom weights
    def mocked_combine_cost_functions(costs, keys, weights_dict=None):
        # We completely ignore the internal default and inject our grid-search weights
        return original_combine_cost_functions(
            costs, keys, weights_dict=custom_weights
        )

    # Monkey-patch the module to use our wrapped function
    sync_module.combine_cost_functions = mocked_combine_cost_functions

    try:
        # Run the sync process normally - it will now use your custom weights on bad_game!
        bad_game.synchronise_tracking_and_event_data()

        # Calculate your performance metrics from the bad_game object
        mean_certainty = bad_game.tracking_data["sync_certainty"].mean()
        variance_certainty = bad_game.tracking_data["sync_certainty"].var()

        # Store results
        result_entry = custom_weights.copy()
        result_entry["mean_certainty"] = mean_certainty
        result_entry["variance_certainty"] = variance_certainty
        results.append(result_entry)

    except Exception as e:
        print(f"Iteration {i+1} failed with error: {e}")

# --- STEP 2C: Clean up and Restore original behavior ---
sync_module.combine_cost_functions = original_combine_cost_functions
print("\n--- GRID SEARCH COMPLETED & ORIGINAL FUNCTION RESTORED ---")

# Convert results to a DataFrame for easy sorting
df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="mean_certainty", ascending=False)
print("\nTop 5 weight combinations for poorly synced events:")
print(df_results.head())

Total grid search iterations to run on bad syncs: 120
Running iteration 1/120 with weights: {'time_cost': 0.25, 'ball_event_dist_cost': 1.0, 'ball_player_dist_cost': 0.5, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 1.0, 'goal_angle_cost': 1.0}


Running iteration 2/120 with weights: {'time_cost': 0.25, 'ball_event_dist_cost': 1.0, 'ball_player_dist_cost': 0.5, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 2.0, 'goal_angle_cost': 1.0}


Running iteration 3/120 with weights: {'time_cost': 0.25, 'ball_event_dist_cost': 1.0, 'ball_player_dist_cost': 1.0, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 1.0, 'goal_angle_cost': 1.0}


Running iteration 4/120 with weights: {'time_cost': 0.25, 'ball_event_dist_cost': 1.0, 'ball_player_dist_cost': 1.0, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 2.0, 'goal_angle_cost': 1.0}


Running iteration 5/120 with weights: {'time_cost': 0.25, 'ball_event_dist_cost': 1.0, 'ball_player_dist_cost': 1.5, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 1.0, 'goal_angle_cost': 1.0}


...

Running iteration 117/120 with weights: {'time_cost': 2.0, 'ball_event_dist_cost': 2.0, 'ball_player_dist_cost': 1.5, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 1.0, 'goal_angle_cost': 1.0}


Running iteration 118/120 with weights: {'time_cost': 2.0, 'ball_event_dist_cost': 2.0, 'ball_player_dist_cost': 1.5, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 2.0, 'goal_angle_cost': 1.0}


Running iteration 119/120 with weights: {'time_cost': 2.0, 'ball_event_dist_cost': 2.0, 'ball_player_dist_cost': 2.0, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 1.0, 'goal_angle_cost': 1.0}


Running iteration 120/120 with weights: {'time_cost': 2.0, 'ball_event_dist_cost': 2.0, 'ball_player_dist_cost': 2.0, 'ball_acc_cost': 1.0, 'player_ball_dist_inc_cost': 2.0, 'goal_angle_cost': 1.0}



--- GRID SEARCH COMPLETED & ORIGINAL FUNCTION RESTORED ---

Top 5 weight combinations for poorly synced events:
    time_cost  ball_event_dist_cost  ball_player_dist_cost  ball_acc_cost  \
1        0.25                   1.0                    0.5            1.0   
3        0.25                   1.0                    1.0            1.0   
9        0.25                   1.5                    0.5            1.0   
25       0.50                   1.0                    0.5            1.0   
5        0.25                   1.0                    1.5            1.0   

    player_ball_dist_inc_cost  goal_angle_cost  mean_certainty  \
1                         2.0              1.0        0.722889   
3                         2.0              1.0        0.704314   
9                         2.0              1.0        0.695718   
25                        2.0              1.0        0.689479   
5                         2.0              1.0        0.688611   

    variance_certainty  
1 

In [12]:
pd.set_option('display.max_columns', None)
df_results.sort_values(by = 'mean_certainty', ascending = False).head(20)

,time_cost,ball_event_dist_cost,ball_player_dist_cost,ball_acc_cost,player_ball_dist_inc_cost,goal_angle_cost,mean_certainty,variance_certainty
1,0.25,1.0,0.5,1.0,2.0,1.0,0.722889,0.013110
3,0.25,1.0,1.0,1.0,2.0,1.0,0.704314,0.021210
9,0.25,1.5,0.5,1.0,2.0,1.0,0.695718,0.022320
25,0.50,1.0,0.5,1.0,2.0,1.0,0.689479,0.007801
5,0.25,1.0,1.5,1.0,2.0,1.0,0.688611,0.030221
11,0.25,1.5,1.0,1.0,2.0,1.0,0.681189,0.030466
7,0.25,1.0,2.0,1.0,2.0,1.0,0.680699,0.037423
0,0.25,1.0,0.5,1.0,1.0,1.0,0.675376,0.021146
27,0.50,1.0,1.0,1.0,2.0,1.0,0.674001,0.013811
13,0.25,1.5,1.5,1.0,2.0,1.0,0.672598,0.037694


In [16]:
import numpy as np
import pandas as pd

# Set display options as before
pd.set_option('display.max_columns', None)

# =====================================================================
# 1. EXTRACT THE WINNING WEIGHTS FROM THE GRID SEARCH RESULTS
# =====================================================================
# Since df_results was sorted descending by mean_certainty, the 1st row is the best
best_run = df_results.iloc[0]

best_weights = {
    "time_cost": best_run["time_cost"],
    "ball_event_dist_cost": best_run["ball_event_dist_cost"],
    "ball_player_dist_cost": best_run["ball_player_dist_cost"],
    "ball_acc_cost": best_run["ball_acc_cost"],
    "player_ball_dist_inc_cost": best_run["player_ball_dist_inc_cost"],
    "goal_angle_cost": best_run["goal_angle_cost"]
}

print("--- OPTIMAL WEIGHTS FOUND ---")
for weight_name, weight_value in best_weights.items():
    print(f"{weight_name}: {weight_value}")

# =====================================================================
# 2. APPLY THE OPTIMAL WEIGHTS AND RUN A FINAL SYNC ON BAD_GAME
# =====================================================================
# Create a wrapper function fixed to our best parameters
def optimal_combine_cost_functions(costs, keys, weights_dict=None):
    return original_combine_cost_functions(costs, keys, weights_dict=best_weights)

# Monkey-patch the module to use the optimal function
sync_module.combine_cost_functions = optimal_combine_cost_functions

try:
    # Synchronize the bad_game subset using the best weights
    bad_game.synchronise_tracking_and_event_data()
    print("\n--- FINAL OPTIMIZED SYNCHRONIZATION COMPLETED ---")
finally:
    # Always restore original framework behavior to avoid side effects later
    sync_module.combine_cost_functions = original_combine_cost_functions


# =====================================================================
# 3. EXTRACT RESULTS AND CALCULATE OPTIMIZED DISTANCES
# =====================================================================
# Copy the subset of events from bad_game
optimized_events_df = bad_game.event_data.copy()

# FIX: Drop the old, outdated certainty column so the new tracking-based one doesn't get suffixed
if "sync_certainty" in optimized_events_df.columns:
    optimized_events_df = optimized_events_df.drop(columns=["sync_certainty"])

# Extract tracking frames using the optimized bad_game object
def extract_optimal_frame(event_id):
    try:
        return bad_game.get_event_frame(event_id)["frame"].iloc[0]
    except (IndexError, ValueError):
        return np.nan

optimized_events_df["tracking_frame"] = optimized_events_df["event_id"].apply(extract_optimal_frame)
optimized_events_df["tracking_frame"] = optimized_events_df["tracking_frame"].astype("Int64")

# Drop any that still couldn't find matches
optimized_events_df = optimized_events_df.dropna(subset=["tracking_frame"])
optimized_events_df["tracking_frame"] = optimized_events_df["tracking_frame"].astype(int)

# Extract optimized ball positions and certainty values from tracking data
optimized_tracking_df = bad_game.tracking_data.copy()
ball_tracking = optimized_tracking_df[["frame", "ball_x", "ball_y", "sync_certainty"]].rename(
    columns={"frame": "tracking_frame"}
)

# Merge tracking frame attributes into the optimized events dataframe
merged_results_df = pd.merge(optimized_events_df, ball_tracking, on="tracking_frame", how="left")
merged_results_df = merged_results_df.loc[:, ~merged_results_df.columns.duplicated(keep='last')]

# Calculate the new spatial discrepancy under the optimized weights
merged_results_df["ball_to_event_distance"] = np.sqrt(
    (merged_results_df["ball_x"] - merged_results_df["start_x"]) ** 2 +
    (merged_results_df["ball_y"] - merged_results_df["start_y"]) ** 2
)

# =====================================================================
# 4. PRINT RE-SYNCHRONIZED METRICS
# =====================================================================
print(f"\nTotal Flagged Events Evaluated: {len(merged_results_df)}")

# Column visibility list 
target_columns = [
    "tracking_frame", "databallpy_event", "start_x", "start_y", 
    "ball_x", "ball_y", "ball_to_event_distance", "sync_certainty",
    "ball_event_dist_cost", "ball_player_dist_cost", "time_cost"
]

# Safeguard in case specific raw cost matrix columns are absent from the schema
available_columns = [col for col in target_columns if col in merged_results_df.columns]

# Display final results sorted by updated distance descending
merged_results_df[available_columns].sort_values(by='ball_to_event_distance', ascending=False)

--- OPTIMAL WEIGHTS FOUND ---
time_cost: 0.25
ball_event_dist_cost: 1.0
ball_player_dist_cost: 0.5
ball_acc_cost: 1.0
player_ball_dist_inc_cost: 2.0
goal_angle_cost: 1.0



--- FINAL OPTIMIZED SYNCHRONIZATION COMPLETED ---

Total Flagged Events Evaluated: 13


,tracking_frame,databallpy_event,start_x,start_y,ball_x,ball_y,ball_to_event_distance,sync_certainty,ball_event_dist_cost,ball_player_dist_cost,time_cost
4,106579,pass,25.514286,33.961556,7.16,-15.07,52.354305,0.673053,1.0,1.0,0.096129
3,106059,pass,44.204167,15.439167,25.29,-31.54,50.643734,0.606740,1.0,1.0,0.047381
5,113095,pass,-4.163250,-2.882250,42.06,-7.81,46.485176,0.631478,1.0,1.0,1.0
12,161582,pass,-14.162857,-25.888222,-47.04,-0.43,41.581578,0.762149,1.0,0.000096,0.025308
0,11128,pass,24.022857,-33.961556,-5.93,-16.80,34.520901,0.734634,1.0,0.001241,1.0
1,31339,pass,-26.922857,21.274889,5.95,28.94,33.754684,0.607926,1.0,1.0,0.709509
8,147183,pass,28.580000,21.736222,41.53,6.25,20.187263,0.621730,1.0,1.0,1.0
2,101186,pass,39.162500,11.497500,32.64,20.07,10.771758,0.627358,1.0,1.0,1.0
11,160723,pass,-29.822857,-6.089167,-30.10,-10.88,4.798843,0.656644,0.002458,0.06518,1.0
6,146867,dribble,48.420833,17.730833,44.72,19.30,4.019758,0.857106,0.00005,0.000029,1.0


In [17]:
# =====================================================================
# 5. MERGE OPTIMIZED RESULTS BACK INTO THE ORIGINAL GAME OBJECT
# =====================================================================

print("\n--- MERGING OPTIMIZED SYNCS BACK INTO ORIGINAL GAME OBJECT ---")

# 1. Ensure the original game.event_data has a baseline 'tracking_frame' column
if "tracking_frame" not in game.event_data.columns:
    game.event_data["tracking_frame"] = game.event_data["event_id"].apply(extract_frame)
game.event_data["tracking_frame"] = game.event_data["tracking_frame"].astype("Int64")

# 2. Create a mapping of event_id -> new tracking_frame from our optimized pass
optimized_frame_map = dict(zip(optimized_events_df["event_id"], optimized_events_df["tracking_frame"]))

# 3. Update the tracking_frame column in the original game.event_data
# If an event_id was optimized, it takes the new frame; otherwise, it keeps its original frame
game.event_data["tracking_frame"] = (
    game.event_data["event_id"]
    .map(optimized_frame_map)
    .fillna(game.event_data["tracking_frame"])
    .astype("Int64")
)

# 4. Update the sync_certainty column in the original game.tracking_data
# Find the specific tracking frames that were updated during our optimized pass
target_frames = optimized_events_df["tracking_frame"].dropna().unique()

# Extract the new certainty values from the bad_game tracking dataset
optimized_certainties = bad_game.tracking_data.loc[
    bad_game.tracking_data["frame"].isin(target_frames), ["frame", "sync_certainty"]
].set_index("frame")["sync_certainty"]

# Map those updated certainties back to the original game tracking dataframe
game.tracking_data.loc[game.tracking_data["frame"].isin(target_frames), "sync_certainty"] = (
    game.tracking_data["frame"]
    .map(optimized_certainties)
    .fillna(game.tracking_data["sync_certainty"])
)

print("Merge complete! The original 'game' object now contains the optimized sync data.")


--- MERGING OPTIMIZED SYNCS BACK INTO ORIGINAL GAME OBJECT ---
Merge complete! The original 'game' object now contains the optimized sync data.
